In [ ]:
import pandas as pd
import numpy as np
import re
import joblib
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import contractions

In [ ]:
print("\nLoading cleaned dataset...")
data = pd.read_csv("final_stress_dataset.csv")
print(f"Loaded {len(data):,} samples")
print("Dataset already cleaned - skipping text preprocessing!")

In [ ]:
# Prepare data
print("\nPreparing data for training...")
X = data["text"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train):,}")
print(f"Test samples: {len(X_test):,}")


In [ ]:
# Tokenization
print("\nTokenizing text...")
MAX_WORDS = 30000  # Increased vocabulary size
MAX_LEN = 80  # Increased sequence length for more context

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_seq = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_seq = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

y_train = np.array(y_train)
y_test = np.array(y_test)

print(f"Vocabulary size: {min(MAX_WORDS, len(tokenizer.word_index) + 1):,}")
print(f"Training shape: {X_train_seq.shape}")
print(f"Test shape: {X_test_seq.shape}")

In [ ]:

# Build Improved LSTM model
print("\nBuilding Enhanced LSTM model...")
vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=256, input_length=MAX_LEN),
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64, return_sequences=False),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print(model.summary())

In [ ]:

# Train the model
print("\nTraining LSTM model...")
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
    min_delta=0.0005
)

In [ ]:
history = model.fit(
    X_train_seq,
    y_train,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

print("\nTraining complete!")

In [ ]:
# Evaluate model
print("\n" + "=" * 70)
print("EVALUATING MODEL")
print("=" * 70)

y_prob = model.predict(X_test_seq, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Non-Stress', 'Stress']))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(f"              Predicted Non-Stress  Predicted Stress")
print(f"Actual Non-Stress     {cm[0][0]:6d}          {cm[0][1]:6d}")
print(f"Actual Stress         {cm[1][0]:6d}          {cm[1][1]:6d}")


In [ ]:
# Save model
print("\n" + "=" * 70)
print("SAVING MODEL")
print("=" * 70)

print("\nSaving stress_lstm_model.keras...")
model.save("stress_lstm_model.keras")

print("Saving tokenizer.pkl...")
joblib.dump(tokenizer, "tokenizer.pkl", compress=3)

print("Saving sequence_config.pkl...")
joblib.dump({"max_len": MAX_LEN}, "sequence_config.pkl", compress=3)